In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#nltk
from nltk.corpus import stopwords

In [5]:
corpus = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?"
]

stopwords = set(stopwords.words('english'))
print(stopwords)


{'doesn', "mustn't", "she'll", 'what', 'your', 'be', 'himself', 'when', 'their', 'my', "they'd", "that'll", "we're", "wasn't", 'above', "he'd", 'needn', 'further', 'he', "they're", 'hadn', 'down', 'where', 'other', "he'll", 'until', 'each', "it's", 'from', 'yours', 'should', 'our', "mightn't", 'had', 'can', 'so', 'any', "it'd", 'hasn', 'some', "aren't", "he's", 'its', 'did', "wouldn't", 'will', "i'd", 'whom', 'over', 'out', 'there', 'o', 'won', 'or', 'isn', "they'll", 'the', 's', 'does', 'wasn', "won't", 'once', 'before', 'being', "shan't", 'these', 'to', 'very', 'itself', 'mustn', 'doing', 'how', 'mightn', "i've", 'd', 'ours', 'ma', 'theirs', 'i', 'same', "hadn't", 'few', 'have', 'again', 'below', 'such', 'shouldn', 'his', 'no', 'couldn', "i'll", 'all', "needn't", 'an', 'you', "haven't", 'hers', 'in', 'into', "we've", 'between', 'after', 'here', 'are', 'them', "you'll", 'with', 'for', 'ain', 'which', 've', "you've", "should've", 'not', 'while', 'during', 'themselves', 'most', 'herself

In [7]:
unique_words = set(word for doc in corpus for word in doc.lower().split() if word not in stopwords)
print(unique_words)

# lets convert corpus to vector 
len_vector = len(unique_words)
vector = np.zeros((len(corpus), len_vector))
print(vector)
word_to_index = {word: i for i, word in enumerate(unique_words)}
for i, doc in enumerate(corpus):
    for word in doc.lower().split():
        if word in word_to_index:
            vector[i, word_to_index[word]] = 1


{'third', 'document?', 'first', 'document', 'one.', 'second', 'document.'}
[[0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]]


In [9]:
vector

array([[0., 0., 1., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0., 1., 1.],
       [1., 0., 0., 0., 1., 0., 0.],
       [0., 1., 1., 0., 0., 0., 0.]])

In [10]:
# there is lot of issue with BOW
# 1. High dimensionality = why ? because each unique word becomes a feature
# 2. Sparsity = most of the entries in the document-term matrix are zeros
# 3. Lack of semantic meaning eg. good vs best

#  High dimensionality can be solve via bi-grams or n-grams


In [12]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 2), stop_words='english')
# here analyzer means we are using word level tokenization and considering both unigrams and bigrams
X = vectorizer.fit_transform(corpus)
X.toarray()


array([[1, 0, 0, 0],
       [2, 1, 1, 1],
       [0, 0, 0, 0],
       [1, 0, 0, 0]])

In [16]:
# tfidf 

movies_reviews = [
    "I loved the movie.",
    "The movie was great!",
    "I did not like the movie.",
    "The film was fantastic.",
    "I hated the film."
]
unique_words = set(word for doc in movies_reviews for word in doc.lower().split() if word not in stopwords)


# loved : tf
def terms_frequency(corpus, terms):
    # manual
    tf = []
    for doc in corpus:
        doc_tf = []
        for term in terms:
            doc_tf.append(doc.lower().split().count(term) / len(doc.split()))
        tf.append(doc_tf)
    return tf

def idf(corpus,terms):
    import math
    idf_values = []
    N = len(corpus)
    for term in terms:
        df = sum(1 for doc in corpus if term in doc.lower().split())
        idf_values.append(math.log((N + 1) / (df + 1)) + 1)  # adding 1 to avoid division by zero
    return idf_values

tf_values = terms_frequency(movies_reviews, unique_words)
idf_values = idf(movies_reviews, unique_words)
idf_values

[2.09861228866811,
 2.09861228866811,
 2.09861228866811,
 2.09861228866811,
 2.09861228866811,
 1.6931471805599454,
 2.09861228866811,
 2.09861228866811,
 2.09861228866811]

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(movies_reviews)
tfidf_values = X.toarray()
tf_values

[[0.0, 0.25, 0.0, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.25],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.16666666666666666, 0.0, 0.16666666666666666, 0.0],
 [0.25, 0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.25, 0.0, 0.0, 0.0, 0.25, 0.0, 0.0]]

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_20newsgroups

newsgroups = fetch_20newsgroups(subset='train')
X = newsgroups.data
y = newsgroups.target

df = pd.DataFrame(tfidf_values, columns=vectorizer.get_feature_names_out())
# df['target'] = y
df.head(2)

,did,fantastic,film,great,hated,like,loved,movie,not,the,was
0,0.0,0.0,0.0,0.000000,0.0,0.0,0.772536,0.517376,0.0,0.368117,0.000000
1,0.0,0.0,0.0,0.655616,0.0,0.0,0.000000,0.439074,0.0,0.312405,0.528947
